# Mental Health Support Chatbot - Mistral 7B + QLoRA Fine-Tuning

Fine-tunes **Mistral-7B-Instruct-v0.3** using **QLoRA** (4-bit quantization + LoRA adapters)
on the **EmpatheticDialogues** dataset to produce a high-quality empathetic mental health chatbot.

## Why Mistral 7B + QLoRA?
- Mistral 7B already understands empathy, emotions, and conversation from pretraining
- QLoRA fits the 7B model into 16GB VRAM by quantizing to 4-bit
- LoRA only trains ~1-2% of parameters - fast and memory efficient
- Result: GPT-4 class response quality at a fraction of the compute

**Estimated runtime:** ~60-90 min on Kaggle P100 GPU

**Requirements:** GPU P100 enabled + HF_TOKEN secret set (see Section 1)

> IMPORTANT: Settings -> Accelerator -> GPU P100 before running

## 1. Environment Setup

In [ ]:
# Verify GPU - need at least 15GB VRAM for Mistral 7B in 4-bit
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('[OK] GPU detected!')
    print(result.stdout[:500])
else:
    raise RuntimeError('[FAIL] No GPU found. Enable GPU: Settings -> Accelerator -> GPU P100')


In [ ]:
# Install all required packages
# bitsandbytes: 4-bit quantization
# peft: LoRA adapters
# trl: SFTTrainer for instruction fine-tuning
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.41.0,<5.0.0',
    'datasets>=2.18.0,<3.0.0',
    'accelerate>=1.0.0',
    'bitsandbytes>=0.43.0',
    'peft>=0.10.0',
    'trl>=0.8.0',
    'tokenizers>=0.21.0',
    'evaluate>=0.4.0',
], check=True)
print('[OK] Packages installed')


In [ ]:
import os, re, json, logging
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

# Suppress tokenizer parallelism warning
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)
from trl import SFTTrainer, SFTConfig

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

# Output paths
OUTPUT_DIR      = Path('/kaggle/working/mistral_mental_health')
ADAPTER_DIR     = OUTPUT_DIR / 'adapter'
FINAL_MODEL_DIR = OUTPUT_DIR / 'final'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:  ', DEVICE)
print('PyTorch: ', torch.__version__)
if DEVICE == 'cuda':
    print('GPU:     ', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM:     {vram:.1f} GB')
    if vram < 14:
        print('[WARN] Less than 14GB VRAM - may OOM. Use T4 x2 or A100 if available.')


In [ ]:
# HuggingFace Authentication
# Requires HF_TOKEN in Kaggle Secrets (Add-ons -> Secrets)
# Steps: huggingface.co/mistralai/Mistral-7B-Instruct-v0.3 -> Agree
# Then: huggingface.co/settings/tokens -> New Read token -> copy
# In Kaggle: Add-ons -> Secrets -> Add New Secret -> Name: HF_TOKEN

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

import os
os.environ['HF_TOKEN'] = secret_value_0
os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_value_0

from huggingface_hub import login
login(token=secret_value_0, add_to_git_credential=False)
print('[OK] HuggingFace authentication successful')


## 2. Configuration

In [ ]:
# ============================================================
# CONFIGURATION - tuned for Kaggle P100 16GB (~75-90 min)
# ============================================================

# Model
MODEL_NAME     = 'mistralai/Mistral-7B-Instruct-v0.3'
MAX_SAMPLES    = 3000   # P100 budget: 3000 samples finishes in ~75 min
MIN_RESP_WORDS = 8      # filter very short responses
MAX_LENGTH     = 256    # 256 vs 512 doubles throughput on P100
SEED           = 42

# Training - P100 math: ~8-12s/step -> 178 steps x 10s = ~30 min
# 2 epochs = ~60 min + overhead = ~75-90 min total
EPOCHS        = 2
BATCH_SIZE    = 4
GRAD_ACCUM    = 8       # effective batch = 4 * 8 = 32
LEARNING_RATE = 2e-4
WARMUP_RATIO  = 0.05

# QLoRA - 4-bit quantization config
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# LoRA adapter config
LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

set_seed(SEED)
print('[OK] Configuration set')
print(f'  Base model:      {MODEL_NAME}')
print(f'  Max samples:     {MAX_SAMPLES}')
print(f'  Max seq length:  {MAX_LENGTH}')
print(f'  Epochs:          {EPOCHS}')
print(f'  Effective batch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'  LoRA rank:       {LORA_CONFIG.r}')


## 3. Dataset Loading & Exploration

In [ ]:
print('Loading EmpatheticDialogues...')
raw_ds   = load_dataset('empathetic_dialogues', trust_remote_code=True)
train_df = raw_ds['train'].to_pandas()
val_df   = raw_ds['validation'].to_pandas()
print(raw_ds)
print(f'Train rows: {len(train_df):,} | Val rows: {len(val_df):,}')


In [ ]:
# Emotion distribution
emotion_counts = train_df['context'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

emotion_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Emotion Distribution (Train)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

train_df['word_count'] = train_df['utterance'].str.split().str.len()
train_df['word_count'].clip(upper=100).hist(bins=50, ax=axes[1], color='mediumpurple', edgecolor='white')
axes[1].set_title('Utterance Length Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count (clipped at 100)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'data_exploration.png'), bbox_inches='tight')
plt.show()
print(train_df['word_count'].describe().to_string())


## 4. Data Preparation & Mistral Chat Template

In [ ]:
# Mistral-Instruct uses this exact chat format:
# <s>[INST] {system}\n\n{user_message} [/INST] {assistant_response}</s>
# We embed the system prompt into every user turn

SYSTEM_PROMPT = (
    'You are a compassionate mental health support assistant. '
    'You listen carefully, respond with empathy, and provide gentle supportive guidance. '
    'You never judge and always validate the person\'s feelings. '
    'You ask thoughtful follow-up questions to better understand their situation.'
)

def format_mistral_chat(user_msg, bot_msg):
    """Format a conversation pair using Mistral instruct template.
    
    Format: <s>[INST] system\n\nuser [/INST] assistant</s>
    The model is only trained to predict tokens AFTER [/INST]
    """
    return (
        f'<s>[INST] {SYSTEM_PROMPT}\n\n{user_msg.strip()} [/INST] '
        f'{bot_msg.strip()}</s>'
    )

def build_dataset(max_samples=None):
    conversations, seen = [], set()
    for split in ['train', 'validation']:
        df = raw_ds[split].to_pandas()
        for _conv_id, group in df.groupby('conv_id'):
            group = group.sort_values('utterance_idx')
            utts  = group['utterance'].tolist()
            for i in range(0, len(utts) - 1, 2):
                u = utts[i].replace('_comma_', ',')
                b = utts[i + 1].replace('_comma_', ',')
                if len(b.split()) < MIN_RESP_WORDS:
                    continue
                key = (u, b)
                if key in seen:
                    continue
                seen.add(key)
                conversations.append({
                    'text':    format_mistral_chat(u, b),
                    'emotion': group['context'].iloc[0],
                    'user':    u,
                    'response': b,
                })
    print(f'Total pairs: {len(conversations):,}')
    if max_samples:
        conversations = conversations[:max_samples]
        print(f'Capped to {max_samples:,}')
    return Dataset.from_list(conversations)

full_ds = build_dataset(MAX_SAMPLES)
print(full_ds)
print('\nSample training text:')
print(full_ds[0]['text'][:400])


In [ ]:
# Train / eval split
split_ds  = full_ds.train_test_split(test_size=0.05, seed=SEED)
train_raw = split_ds['train']
eval_raw  = split_ds['test']
print(f'Train: {len(train_raw):,}  Eval: {len(eval_raw):,}')


## 5. Tokenizer Setup

In [ ]:
print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=os.environ['HF_TOKEN'],
    padding_side='right',   # CRITICAL: must be right for SFTTrainer with causal LM
)

# Mistral tokenizer has no pad token by default - use eos token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f'Vocab size:  {len(tokenizer):,}')
print(f'Pad token:   {tokenizer.pad_token!r}  (id={tokenizer.pad_token_id})')
print(f'EOS token:   {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})')
print(f'BOS token:   {tokenizer.bos_token!r}  (id={tokenizer.bos_token_id})')

# Verify the chat template looks correct
sample = full_ds[0]['text']
tokens = tokenizer(sample, return_tensors='pt')
print(f'\nSample token count: {tokens["input_ids"].shape[1]}')
print(f'(MAX_LENGTH is {MAX_LENGTH} - sequences longer than this will be truncated)')


## 6. Load Model in 4-bit (QLoRA)

In [ ]:
print('Loading Mistral 7B in 4-bit quantized mode...')
print('This downloads ~14GB and loads into ~5.5GB VRAM - takes 3-5 minutes...')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BNB_CONFIG,
    device_map='auto',          # automatically place layers on GPU
    token=os.environ['HF_TOKEN'],
    torch_dtype=torch.float16, # bf16 for non-quantized layers
    attn_implementation='eager', # use standard attention (flash_attn not always available)
)

# Required for QLoRA: enables gradient checkpointing + casts layer norms to float32
model = prepare_model_for_kbit_training(model)

# Disable cache - incompatible with gradient checkpointing
model.config.use_cache = False
model.config.pretraining_tp = 1

print('[OK] Base model loaded and prepared for QLoRA')

# Memory report
if DEVICE == 'cuda':
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM used:      {allocated:.1f} GB')
    print(f'VRAM reserved:  {reserved:.1f} GB')
    print(f'VRAM total:     {total:.1f} GB')
    print(f'VRAM free:      {total - reserved:.1f} GB')


In [ ]:
# Apply LoRA adapters to the model
model = get_peft_model(model, LORA_CONFIG)

# Print trainable parameter count
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct = 100 * trainable_params / total_params

print('[OK] LoRA adapters applied')
print(f'  Total parameters:     {total_params:,}')
print(f'  Trainable parameters: {trainable_params:,}  ({pct:.2f}%)')
print(f'  Frozen parameters:    {total_params - trainable_params:,}')
print()
print('LoRA target modules:')
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f'  {name}: {param.shape}')
        if 'lora' in name.lower():
            break  # just show first few to confirm


## 7. Fine-Tuning with SFTTrainer

In [ ]:
# SFTConfig extends TrainingArguments with SFT-specific options
sft_config = SFTConfig(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_ratio                = WARMUP_RATIO,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = 'cosine',
    weight_decay                = 0.001,

    # Precision - bf16 is better than fp16 for Mistral
    fp16                        = True,
    bf16                        = False,

    # Memory optimizations
    gradient_checkpointing      = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    optim                       = 'paged_adamw_32bit', # QLoRA standard optimizer
    dataloader_num_workers      = 0,

    # Logging & saving
    logging_steps               = 25,
    eval_strategy               = 'steps',
    eval_steps                  = 50 ,
    save_strategy               = 'steps',
    save_steps                  = 50 ,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = 'none',
    seed                        = SEED,

    # SFT specific
    max_seq_length              = MAX_LENGTH,
    dataset_text_field          = 'text',  # column in our dataset
    packing                     = False,   # dont pack sequences - cleaner for chat data
)

trainer = SFTTrainer(
    model           = model,
    args            = sft_config,
    train_dataset   = train_raw,
    eval_dataset    = eval_raw,
    processing_class = tokenizer,
)

print('[OK] SFTTrainer configured')
steps_per_epoch = len(train_raw) // (BATCH_SIZE * GRAD_ACCUM)
print(f'  Effective batch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'  Steps/epoch:     {steps_per_epoch}')
print(f'  Total steps:     {steps_per_epoch * EPOCHS}')
print(f'  Eval every:      50 steps')


In [ ]:
print('Starting QLoRA fine-tuning...')
print('Expected time: ~60-90 minutes on P100')
print('You will see loss logs every 25 steps')
print()

# Clear CUDA cache before training
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

train_result = trainer.train()

print('\n[OK] Training complete!')
print(f'  Final train loss: {train_result.training_loss:.4f}')
best = trainer.state.best_metric
print(f'  Best eval loss:   {best:.4f}' if best is not None else '  Best eval loss:   N/A (no eval ran)')
print(f'  Total steps:      {train_result.global_step}')


## 8. Save Adapter & Merge

In [ ]:
# Step 1: Save the LoRA adapter (lightweight - ~50-100MB)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f'[OK] LoRA adapter saved to: {ADAPTER_DIR}')
print('Adapter files:')
for fp in sorted(ADAPTER_DIR.iterdir()):
    print(f'  {fp.name:<40}  {fp.stat().st_size/1e6:.1f} MB')


In [ ]:
# Step 2: Merge LoRA adapter into base model for standalone deployment
# merge_and_unload() dequantizes ONE layer at a time (not all at once)
# Peak VRAM during merge is only ~4-5GB - safe on P100
# NOTE: model.cpu() would CRASH - bitsandbytes 4-bit layers are CUDA-only
print('Merging LoRA adapter into base model...')
print('(Takes 3-5 minutes - dequantizes layer by layer)')

if DEVICE == 'cuda':
    torch.cuda.empty_cache()

# Merge LoRA weights into base model (runs on GPU, layer by layer)
merged_model = model.merge_and_unload()

# Save merged model
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(str(FINAL_MODEL_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

# Save training metadata
best_metric = trainer.state.best_metric
info = {
    'base_model':     MODEL_NAME,
    'lora_rank':      LORA_CONFIG.r,
    'lora_alpha':     LORA_CONFIG.lora_alpha,
    'target_modules': list(LORA_CONFIG.target_modules),
    'epochs':         EPOCHS,
    'max_samples':    MAX_SAMPLES,
    'max_length':     MAX_LENGTH,
    'learning_rate':  LEARNING_RATE,
    'batch_size':     BATCH_SIZE,
    'grad_accum':     GRAD_ACCUM,
    'train_loss':     train_result.training_loss,
    'best_eval_loss': round(best_metric, 4) if best_metric is not None else None,
    'total_steps':    train_result.global_step,
}
with open(OUTPUT_DIR / 'training_info.json', 'w', encoding='utf-8') as f:
    json.dump(info, f, indent=2)

print(f'[OK] Merged model saved to: {FINAL_MODEL_DIR}')
print('Saved files:')
for fp in sorted(FINAL_MODEL_DIR.iterdir()):
    print(f'  {fp.name:<40}  {fp.stat().st_size/1e6:.1f} MB')


## 9. Training Curves

In [ ]:
log_history  = trainer.state.log_history
train_steps  = [x['step'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
train_losses = [x['loss'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_steps   = [x['step'] for x in log_history if 'eval_loss' in x]
eval_losses  = [x['eval_loss'] for x in log_history if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train_steps, train_losses, label='Train Loss', color='steelblue', alpha=0.7, linewidth=1.5)
ax.plot(eval_steps, eval_losses, label='Eval Loss', color='tomato', linewidth=2.5, marker='o', markersize=6)
ax.set_xlabel('Training Steps', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Mistral 7B QLoRA Fine-Tuning Loss', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), bbox_inches='tight')
plt.show()

if eval_losses:
    best_idx = eval_losses.index(min(eval_losses))
    print(f'Best eval loss: {eval_losses[best_idx]:.4f} at step {eval_steps[best_idx]}')


## 10. Inference Test

In [ ]:
# Free all training memory before loading inference model
print('Freeing training memory...')
try:
    del trainer
except NameError: pass
try:
    del model
except NameError: pass
try:
    del merged_model
except NameError: pass
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    import gc; gc.collect()

# Load the merged model for inference
print('Loading merged model for inference...')

inf_tok = AutoTokenizer.from_pretrained(str(FINAL_MODEL_DIR))
if inf_tok.pad_token is None:
    inf_tok.pad_token = inf_tok.eos_token

# Reload in 4-bit for memory efficiency
inf_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
inf_model = AutoModelForCausalLM.from_pretrained(
    str(FINAL_MODEL_DIR),
    quantization_config=inf_bnb,
    device_map='auto',
    torch_dtype=torch.float16,
)
# Re-enable KV cache for fast inference (was disabled during training)
inf_model.config.use_cache = True
inf_model.eval()
print('[OK] Inference model loaded')


In [ ]:
def generate_response(
    user_message,
    max_new_tokens=300,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
):
    """Generate an empathetic response using Mistral chat template."""
    prompt = f'<s>[INST] {SYSTEM_PROMPT}\n\n{user_message.strip()} [/INST]'

    inputs = inf_tok(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = inf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            do_sample=True,
            pad_token_id=inf_tok.pad_token_id,
            eos_token_id=inf_tok.eos_token_id,
        )

    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    response = inf_tok.decode(new_tokens, skip_special_tokens=True).strip()
    return response if response else '[No response generated]'


TEST_PROMPTS = [
    'I have been feeling really anxious about my job interview tomorrow.',
    'I feel so alone lately. Nobody really understands what I am going through.',
    'I cannot stop overthinking everything and it is exhausting.',
    'I have been really sad and I do not know why.',
    'I am so stressed out from work and I do not know how to cope.',
]

print('=' * 70)
print('INFERENCE TEST - Mistral 7B QLoRA Mental Health Chatbot')
print('=' * 70)
for prompt in TEST_PROMPTS:
    resp = generate_response(prompt)
    print(f'\nUser:      {prompt}')
    print(f'Assistant: {resp}')
    print('-' * 70)


## 11. Quick Evaluation

In [ ]:
EMPATHY_POS = [
    'understand', 'hear', 'feel', 'valid', 'normal', 'okay', 'support',
    'here', 'listen', 'care', 'difficult', 'hard', 'together', 'help',
    'courage', 'share', 'strength', 'important', 'matter', 'sorry',
    'sense', 'natural', 'completely', 'reach out', 'not alone',
]
EMPATHY_NEG = ['you should', 'just', 'simply', 'easy', 'stop', 'dont worry', 'get over', 'move on']

def empathy_score(text):
    t = text.lower()
    pos = sum(1 for w in EMPATHY_POS if w in t)
    neg = sum(1 for w in EMPATHY_NEG if w in t)
    return max(0.0, min(1.0, (pos - neg * 2) / len(EMPATHY_POS)))

EVAL_SET = [
    ('I have been feeling really anxious about my interview.',   'anxious'),
    ('I feel so alone. Nobody seems to care about me.',          'lonely'),
    ('I cannot stop crying and I do not know why.',              'sad'),
    ('Everything feels overwhelming right now.',                 'overwhelmed'),
    ('I had a panic attack at work today.',                      'anxious'),
    ('I have been really down and nothing brings me joy.',       'depressed'),
    ('I am so angry at myself for making that mistake.',         'angry'),
    ('I cannot sleep, my mind will not stop racing.',            'anxious'),
    ('I feel like a burden to everyone around me.',              'sad'),
    ('I am exhausted but I cannot rest, too much to do.',        'stressed'),
]

results, all_resp = [], []
print(f'{"Emotion":<12} {"Empathy":>8} {"Words":>6}   Response preview')
print('-' * 80)

for prompt, emotion in EVAL_SET:
    resp  = generate_response(prompt)
    score = empathy_score(resp)
    words = len(resp.split())
    all_resp.append(resp)
    results.append({'emotion': emotion, 'score': score, 'words': words, 'response': resp})
    print(f'{emotion:<12} {score:>8.2f} {words:>6}   {resp[:50]}...')

all_tokens = ' '.join(all_resp).lower().split()
ttr        = len(set(all_tokens)) / max(len(all_tokens), 1)
avg_score  = np.mean([r['score'] for r in results])
avg_words  = np.mean([r['words'] for r in results])

print('\n' + '=' * 80)
print('EVALUATION SUMMARY')
print('=' * 80)
print(f'  Average Empathy Score:    {avg_score:.3f} / 1.000')
print(f'  Response Diversity (TTR): {ttr:.3f}')
print(f'  Avg Response Length:      {avg_words:.0f} words')

with open(OUTPUT_DIR / 'evaluation_results.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model': MODEL_NAME,
        'avg_empathy': round(avg_score, 3),
        'ttr': round(ttr, 3),
        'avg_words': round(avg_words, 1),
        'per_emotion': results
    }, f, indent=2)
print(f'[OK] Evaluation saved')


## 12. Output Summary & Download Instructions

In [ ]:
print('=' * 60)
print('OUTPUT FILES')
print('=' * 60)
total_size = 0
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        sz = p.stat().st_size
        total_size += sz
        print(f'  {str(p.relative_to(OUTPUT_DIR)):<45}  {sz/1e6:>7.1f} MB')
print(f'\n  Total: {total_size/1e9:.2f} GB')

print()
print('=' * 60)
print('DOWNLOAD OPTIONS')
print('=' * 60)
options = [
    'OPTION A - Adapter only (recommended, small ~100MB):',
    '  Download: mistral_mental_health/adapter/',
    '  To use locally:',
    '    from peft import PeftModel',
    '    base = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3", ...)',
    '    model = PeftModel.from_pretrained(base, "path/to/adapter")',
    '',
    'OPTION B - Merged model (standalone, large ~14GB):',
    '  Download: mistral_mental_health/final/',
    '  To use locally:',
    '    model = AutoModelForCausalLM.from_pretrained("path/to/final", ...)',
    '    (No peft library needed)',
    '',
    'LOCAL USAGE:',
    '  Place downloaded files in: mental-health-chatbot/models/mistral_mental_health/',
    '  Update MODEL_PATH in src/chatbot.py to point to this folder',
    '  Run: streamlit run app.py',
]
for line in options:
    print(' ', line)
